# Customer Churn Prediction System
**Course/Project Portfolio Project**

## Project Objective
Predict whether a customer is likely to churn (leave the telecom company) based on customer account info, subscription details, and billing behavior. 

This notebook demonstrates a modular, production-style machine learning workflow using **Scikit-Learn**, **Pandas**, **Seaborn**, and three classification algorithms:
1. **Logistic Regression**
2. **Decision Tree Classifier**
3. **Random Forest Classifier**

---

## 1. Environment & Setup

First, we import the necessary packages and verify that our system path includes the custom `src/` modules.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add the workspace root to sys.path to allow module imports from src
sys.path.append(os.path.abspath('..'))

# Import custom src modules
from src.data_cleaning import load_data, clean_data
from src.feature_engineering import preprocess_data, get_preprocessor
from src.visualization import CHURN_PALETTE
from src.model_training import split_dataset, train_models
from src.evaluation import evaluate_model, compare_models

%matplotlib inline

## 2. Data Loading & Cleaning

We use `load_data()` which automatically downloads the public **Telco Customer Churn** dataset (originally IBM) from our public source, and then clean the data using `clean_data()`.

In [ ]:
# Load raw data
df_raw = load_data()
print(f"Raw Shape: {df_raw.shape}")

In [ ]:
# Inspect missing values and basic info
df_raw.info()

In [ ]:
# Clean the dataset
df_clean = clean_data(df_raw)
df_clean.head()

## 3. Exploratory Data Analysis (EDA)

Let's explore key drivers of churn visually. We'll plot churn distribution, and cross-examine churn rates with Contract, Monthly Charges, and Tenure.

In [ ]:
# Churn distribution countplot
plt.figure(figsize=(6, 4.5))
ax = sns.countplot(x='Churn', data=df_clean, palette=CHURN_PALETTE)
plt.title('Customer Churn Distribution', weight='bold')
plt.xticks([0, 1], ['Active (No)', 'Churned (Yes)'])
plt.ylabel('Number of Customers')

# Annotate percentage
total = len(df_clean)
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{height} ({height/total*100:.1f}%)',
                (p.get_x() + p.get_width() / 2., height / 2),
                ha='center', va='center', color='white', weight='bold')
plt.show()

In [ ]:
# Churn rate by Contract Type
plt.figure(figsize=(8, 4.5))
sns.countplot(x='Contract', hue='Churn', data=df_clean, palette=CHURN_PALETTE)
plt.title('Churn by Contract Type', weight='bold')
plt.legend(labels=['Active', 'Churned'])
plt.ylabel('Customer Count')
plt.show()

In [ ]:
# Distribution of Monthly Charges by Churn Status
plt.figure(figsize=(9, 4.5))
sns.kdeplot(data=df_clean, x='MonthlyCharges', hue='Churn', fill=True, palette=CHURN_PALETTE, common_norm=False, alpha=0.5)
plt.title('Monthly Charges Distribution by Churn Status', weight='bold')
plt.xlabel('Monthly Charges ($)')
plt.legend(labels=['Churned', 'Active'])
plt.show()

In [ ]:
# Tenure boxplot vs Churn
plt.figure(figsize=(8, 4.5))
sns.boxplot(x='Churn', y='tenure', data=df_clean, palette=CHURN_PALETTE)
plt.title('Customer Tenure vs Churn Status', weight='bold')
plt.xticks([0, 1], ['Active', 'Churned'])
plt.xlabel('Churn Status')
plt.ylabel('Tenure (Months)')
plt.show()

In [ ]:
# Correlation heatmap for numerical features
plt.figure(figsize=(7, 5))
sns.heatmap(df_clean.select_dtypes(include=[np.number]).corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix of Numerical Features', weight='bold')
plt.show()

## 4. Feature Engineering & Preprocessing

We use `preprocess_data()` to clean columns, bin `tenure` into ranges, and list numerical vs categorical splits. Then, we fetch the ColumnTransformer.

In [ ]:
X, y, cat_cols, num_cols = preprocess_data(df_clean)
preprocessor = get_preprocessor(cat_cols, num_cols)
print(f"X Shape: {X.shape}")

## 5. Model Training & Evaluation

We perform an 80-20 train-test split, build the Pipelines (Preprocessor + Classifier) for three models, and evaluate them on test set metrics.

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = split_dataset(X, y)

# Train all 3 pipeline models
pipelines = train_models(X_train, y_train, preprocessor)

In [ ]:
# Evaluate and collect metrics
eval_results = []
for name, model_pipeline in pipelines.items():
    metrics = evaluate_model(model_pipeline, X_test, y_test, name)
    eval_results.append(metrics)

## 6. Model Comparison & Conclusion

In [ ]:
comparison_df, best_model = compare_models(eval_results)

## 7. Strategic Business Insights Summary

Based on the analysis, here are the main insights and recommended actions:
1. **Contract Strategy**: Month-to-month contracts account for the highest churn volume. Migrating these customers to 1- or 2-year terms via loyalty incentives is crucial.
2. **Billing Strategy**: Electronic check users have much higher churn than auto-pay users. Offering incentives to register for auto-pay will reduce friction.
3. **Price Audits**: Churned customers have higher monthly charges. A service-value audit is recommended for high-bill customer cohorts.
4. **Onboarding focus**: The first 12 months is the peak risk window. Proactive onboarding support should be deployed during this period.